In [1]:
import sys
import os
import requests
import pandas as pd
from io import BytesIO
from datetime import datetime, time

url = "https://1drv.ms/x/c/d6aca2526f83594b/IQAc8ap_zmWhR5goYf70_b69AU0CyhmpxFroY48j8pNaZac?download=1"

def get_data(url):
    response = requests.get(url)
    df = pd.read_excel(BytesIO(response.content), engine="openpyxl")
    return df

df = get_data(url)


print("program ran ... ")

program ran ... 


In [2]:
df_tow = df.copy()
print(df_tow.tail())

     Unnamed: 0 Unnamed: 1  2025-09-01 00:00:00     45901  \
1837        NaN        NaN  2026-04-08 00:00:00  09:45:00   
1838        NaN        NaN  2026-04-08 00:00:00  12:55:00   
1839        NaN        NaN  2026-04-08 00:00:00  16:00:00   
1840        NaN        NaN  2026-04-08 00:00:00  19:12:00   
1841        NaN        NaN  2026-04-08 00:00:00  21:12:00   

     *READINGS ARE FROM TOP OF CASING (TOC)* Unnamed: 5 Unnamed: 6 Unnamed: 7  \
1837                                   30.43      10.37      33.62      38.85   
1838                                   30.48      10.38      33.62      38.84   
1839                                   30.42      10.36      33.64      38.82   
1840                                   30.38      10.35      33.63      38.79   
1841                                   30.39      10.35      33.67      38.83   

     Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12 Unnamed: 13  \
1837      32.03      40.22       41.73       36.21         NaN      

In [3]:
def shape_df(df):
    # cut the first 5 rows and first column 
    df = df.iloc[25:,2:].reset_index(drop=True)
    return df

df_tow = shape_df(df_tow)

print(df_tow.info())

# df_tow.to_excel('tow-test.xlsx', engine="openpyxl")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1817 entries, 0 to 1816
Data columns (total 15 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   2025-09-01 00:00:00                      1817 non-null   object
 1   45901                                    1817 non-null   object
 2   *READINGS ARE FROM TOP OF CASING (TOC)*  1572 non-null   object
 3   Unnamed: 5                               1782 non-null   object
 4   Unnamed: 6                               1688 non-null   object
 5   Unnamed: 7                               1803 non-null   object
 6   Unnamed: 8                               1768 non-null   object
 7   Unnamed: 9                               1662 non-null   object
 8   Unnamed: 10                              1701 non-null   object
 9   Unnamed: 11                              1176 non-null   object
 10  Unnamed: 12                              1103 non-null   obj

In [4]:
print(len(df_tow.columns))
columns = ['date',	'time', 'TOW_1','TOW_2','TOW_3', 'TOW_4', 'TOW_5', 'TOW_6', 'TOW_7', 'TOW_8', 'TOW_15'	,'TOW_16','TOW_17','TOW_18', 'comments']
print(len(columns))
def trim_frame(df, new_col):
    df = df.iloc[:, :len(new_col)] 
    df.columns = new_col
    return df

def clean_col(df):
    # strip all columns with "Unnamed" string
    for col in df.columns:
        if "Unnamed" in str(col):
            df = df.drop(columns=[str(col)], errors='ignore')
    return df

df_tow = trim_frame(df_tow, columns)
df_tow = clean_col(df_tow)
print(df_tow.columns)

15
15
Index(['date', 'time', 'TOW_1', 'TOW_2', 'TOW_3', 'TOW_4', 'TOW_5', 'TOW_6',
       'TOW_7', 'TOW_8', 'TOW_15', 'TOW_16', 'TOW_17', 'TOW_18', 'comments'],
      dtype='object')


In [5]:
def fix_time(df):
    mask = df['time'].apply(lambda x: isinstance(x, datetime))
    df.loc[mask, 'time'] = df.loc[mask, 'time'].apply(lambda x: x.time())
    return df

def make_datetime_col(df):
    df['datetime'] = df.apply(
        lambda row: datetime.combine(row['date'].date(), row['time']),
        axis=1
    )
    return df
def fix_space(df):
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)  # replace spaces
    return df

def drop_DT(df):
    # strip all columns with "Unnamed" string
    df = df.drop(columns=['date', 'time'], errors='ignore')
    return df

df_tow = fix_time(df_tow)
df_tow = make_datetime_col(df_tow)
df_tow = fix_space(df_tow)
df_tow = drop_DT(df_tow)

print(df_tow)

      TOW_1  TOW_2  TOW_3  TOW_4  TOW_5  TOW_6  TOW_7  TOW_8 TOW_15 TOW_16  \
0       NaN    NaN    NaN   8.95    NaN    NaN    NaN    NaN    NaN   6.17   
1       NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   7.45   
2       NaN    NaN    NaN   8.82    NaN    NaN    NaN    NaN    NaN   6.34   
3       NaN    NaN    NaN   8.82    NaN    NaN    NaN    NaN    NaN    6.3   
4       NaN    NaN    NaN   8.77    NaN    NaN    NaN    NaN    NaN   6.45   
...     ...    ...    ...    ...    ...    ...    ...    ...    ...    ...   
1812  30.43  10.37  33.62  38.85  32.03  40.22  41.73  36.21    NaN  41.77   
1813  30.48  10.38  33.62  38.84  31.97  40.25   41.8  36.28   29.6  41.68   
1814  30.42  10.36  33.64  38.82  31.95  40.18  41.26  36.25  29.41  42.02   
1815  30.38  10.35  33.63  38.79  31.93  40.18  41.35  36.23  29.38  42.27   
1816  30.39  10.35  33.67  38.83  31.95  40.24  41.42  36.27  29.37  42.25   

     TOW_17 TOW_18                                           co

In [52]:
print(df_tow.columns)

Index(['TOW_1', 'TOW_2', 'TOW_3', 'TOW_4', 'TOW_5', 'TOW_6', 'TOW_7', 'TOW_8',
       'TOW_15', 'TOW_16', 'TOW_17', 'TOW_18', 'comments', 'datetime'],
      dtype='object')


In [6]:
def make_tidy(df):

    value_cols = [col for col in df.columns if 'TOW_' in col ]

    df_long = df.melt(
        id_vars='datetime',
        value_vars=value_cols,
        var_name='tow',
        value_name='value'
    )
    
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long.columns.name = None
    return df_long

df_long = make_tidy(df_tow)
print(df_long.tail(10))




                 datetime     tow  value
21794 2026-04-07 21:12:00  TOW_18  51.30
21795 2026-04-08 00:00:00  TOW_18  51.33
21796 2026-04-08 02:00:00  TOW_18  51.35
21797 2026-04-08 05:10:00  TOW_18  51.30
21798 2026-04-08 07:00:00  TOW_18  51.36
21799 2026-04-08 09:45:00  TOW_18  51.19
21800 2026-04-08 12:55:00  TOW_18    NaN
21801 2026-04-08 16:00:00  TOW_18    NaN
21802 2026-04-08 19:12:00  TOW_18  51.14
21803 2026-04-08 21:12:00  TOW_18  51.16


In [55]:
phase_1 = 4568
phase_2 = 4543

toc_elev = {
    'TOW_1': 4572.92,
    'TOW_2': 4571.70,
    'TOW_3': 4572.80,
    'TOW_4': 4574.58,
    'TOW_5': 4572.98,
    'TOW_6': 4573.53,
    'TOW_7': 4573.30,
    'TOW_8': 4569.75,
    'TOW_15': 4589.30,
    'TOW_16': 4580.34,
    'TOW_17': 4581.15,
    'TOW_18': 4586.93
}
cur_elev = {
    'TOW_1': 0,
    'TOW_2': 0,
    'TOW_3': 0,
    'TOW_4': 0,
    'TOW_5': 0,
    'TOW_6': 0,
    'TOW_7': 0,
    'TOW_8': 0,
    'TOW_15': 0,
    'TOW_16': 0,
    'TOW_17': 0,
    'TOW_18': 0
}

In [ ]:
# print(toc_elev['TOW_1'])
# latest_df = df.loc[df.groupby('pump')['datetime'].idxmax()].reset_index(drop=True)
# depth = df_long.loc[df_long.groupby('tow',)['datetime'].idxmax()].reset_index(drop=True)

depth = ((df_long[df_long['value'].notna()]).sort_values('datetime').groupby('tow', as_index=False).last())

depth = depth.set_index('tow')


for k in toc_elev.keys():
    top = toc_elev[k]
    drop = (depth.loc[k]['value'])
    cur_elev[k] = (top - drop)

cur_elev = {k: round(v, 2) for k, v in cur_elev.items()}
print(cur_elev)



{'TOW_1': np.float64(4542.53), 'TOW_2': np.float64(4561.35), 'TOW_3': np.float64(4539.13), 'TOW_4': np.float64(4535.75), 'TOW_5': np.float64(4541.03), 'TOW_6': np.float64(4533.29), 'TOW_7': np.float64(4531.88), 'TOW_8': np.float64(4533.48), 'TOW_15': np.float64(4559.93), 'TOW_16': np.float64(4538.09), 'TOW_17': np.float64(4542.85), 'TOW_18': np.float64(4535.77)}
